In [1]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_aptma_directory():
    url = "https://aptma.org.pk/directory-2/"
    
    # Define a realistic user-agent header to avoid getting blocked by security firewalls
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
    }
    
    print(f"Fetching data from {url}...")
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the website: {e}")
        return

    soup = BeautifulSoup(response.text, "html.parser")
    
    # Lists to store the extracted data fields
    extracted_data = []
    
    # Find all textual containers where the directory data resides
    # This checks common structures such as paragraphs, list entries, or table rows
    containers = soup.find_all(['p', 'li', 'tr', 'div', 'td'])
    
    # Regex patterns for validation
    phone_pattern = re.compile(r'(\+?\d[\d\-\s\,]{7,\d})')
    website_pattern = re.compile(r'([a-zA-Z0-9\-\.]+\.(?:com|org|net|pk|info|com\.pk))', re.IGNORECASE)
    
    print("Parsing directory records...")
    
    # Track items to avoid duplicate records from overlapping containers
    seen_entries = set()

    for container in containers:
        text = container.get_text(separator=" ", strip=True)
        
        # APTMA entries usually contain the word "Member." or name prefix "MR. " / "SHEIKH "
        if "Member" in text or "MR." in text or "MS." in text:
            # Clean up redundant spaces
            text_cleaned = re.sub(r'\s+', ' ', text)
            
            if text_cleaned in seen_entries or len(text_cleaned) < 30:
                continue
            seen_entries.add(text_cleaned)
            
            # Initialize empty fields
            company_name = "N/A"
            member_name = "N/A"
            phone_num = "N/A"
            website_url = "N/A"
            
            # --- Field 1: Company Name ---
            # Usually precedes '. Member' or '. MR.'
            company_match = re.search(r'^(.*?)(?:\. Member|\. MR\.|\. MS\.)', text_cleaned)
            if company_match:
                company_name = company_match.group(1).strip()
            
            # --- Field 2: Member Name ---
            # Usually starts with MR. or MS. or SHEIKH
            member_match = re.search(r'((?:MR\.|MS\.|SHEIKH|MIAN)\s+[A-Z\s\.]+?)(?:\.|\s\+?\d|\s[a-z0-9])', text_cleaned)
            if member_match:
                member_name = member_match.group(1).strip()
            elif "Member." in text_cleaned:
                # Fallback if prefix varies
                parts = text_cleaned.split("Member.")
                if len(parts) > 1:
                    sub_parts = parts[1].strip().split(".")
                    if len(sub_parts) > 0:
                        member_name = sub_parts[0].strip()

            # --- Field 3: Phone ---
            phone_matches = phone_pattern.findall(text_cleaned)
            if phone_matches:
                # Clean up extracted string to make sure it looks like a phone number
                for match in phone_matches:
                    match_clean = match.strip(",- ")
                    if len(re.sub(r'\D', '', match_clean)) >= 7:
                        phone_num = match_clean
                        break
            
            # --- Field 4: Website ---
            web_match = website_pattern.search(text_cleaned)
            if web_match:
                website_url = web_match.group(1).lower().strip()
            
            # Only append if we successfully parsed at least the company name
            if company_name != "N/A" and len(company_name) > 3:
                extracted_data.append({
                    "company_name": company_name,
                    "member_name": member_name,
                    "Phone": phone_num,
                    "websites": website_url
                })

    # Convert to a structured DataFrame
    if extracted_data:
        df = pd.DataFrame(extracted_data)
        
        # Deduplicate records based on company name
        df.drop_duplicates(subset=["company_name"], keep="first", inplace=True)
        
        # Exporting to the requested Excel file name
        output_filename = "final_data_aptma.xlsx"
        df.to_excel(output_filename, index=False)
        print(f"Successfully saved {len(df)} records to '{output_filename}'!")
    else:
        print("No directory records were found. Please verify the layout/structure of the target URL.")

if __name__ == "__main__":
    scrape_aptma_directory()

Fetching data from https://aptma.org.pk/directory-2/...
Parsing directory records...
Successfully saved 5 records to 'final_data_aptma.xlsx'!
